In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer

In [13]:
DATASET_PATH = '../Data/Raw/raw_dataset.csv'
TEST_DATASET_PATH = '../Data/Processed/test_dataset.csv'

def load_dataset(path: str):
    if not os.path.exists(path): return None
    return pd.read_csv(path)
    
dataset = load_dataset(DATASET_PATH)
# dataset.drop(columns='Student_ID', inplace=True)
dataset

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,STU1996,23,Male,2,7.5,20.3,1.89,7.3,10.7,High,8.2,6.3,80.9,No,Average,No,Medium
1996,STU1997,21,Male,2,6.4,16.8,2.14,4.6,5.8,Moderate,4.6,3.5,76.9,Yes,Poor,No,High
1997,STU1998,19,Female,1,4.5,13.1,2.54,7.5,10.0,Low,6.7,8.1,87.9,Yes,NaN,No,High
1998,STU1999,21,Male,3,8.1,22.3,2.62,4.5,6.6,Moderate,6.5,8.4,100.0,No,Good,No,Low


In [14]:
def fix_non_real_screen_times(dataset: pd.DataFrame, drop=False):
    mask = dataset['Social_Media_Hours'] > dataset['Screen_Time_Hours']

    if drop:
        indices = dataset[mask].index
        dataset = dataset.drop(indices)
    else:
        tmp = dataset['Screen_Time_Hours'].iloc[mask]
        dataset.loc[mask, 'Screen_Time_Hours'] = dataset['Social_Media_Hours'].iloc[mask]
        dataset.loc[mask, 'Social_Media_Hours'] = tmp

    mask = dataset['Social_Media_Hours'] > dataset['Screen_Time_Hours']
    print(f'Non Logic Screen Time Counts : {mask.sum()}')


fix_non_real_screen_times(dataset)

Non Logic Screen Time Counts : 0


In [16]:
def drop_non_impacting_columns(dataset: pd.DataFrame, columns=[]):
    return dataset.drop(columns=columns)

dataset = drop_non_impacting_columns(dataset, ['Age', 'Gender'])

In [17]:
mask = ((~dataset.isna()) & (~dataset.isnull())).all(axis=1)
print(f'{((~mask).mean() * 100):.2f}% of the data contain null values')

19.70% of the data contain null values


In [18]:
dataset

,Student_ID,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,STU1996,2,7.5,20.3,1.89,7.3,10.7,High,8.2,6.3,80.9,No,Average,No,Medium
1996,STU1997,2,6.4,16.8,2.14,4.6,5.8,Moderate,4.6,3.5,76.9,Yes,Poor,No,High
1997,STU1998,1,4.5,13.1,2.54,7.5,10.0,Low,6.7,8.1,87.9,Yes,NaN,No,High
1998,STU1999,3,8.1,22.3,2.62,4.5,6.6,Moderate,6.5,8.4,100.0,No,Good,No,Low


In [19]:
# # Split the data ~~ 200 rows in test and ~~ 1800 in train

# all_ids = np.full((dataset.shape[0]), 1, dtype=np.uint8)
# dataset = dataset.drop(columns='Student_ID')

# clean_ids = np.where(mask)[0]
# _, test_ids = train_test_split(clean_ids, random_state=45, test_size=0.13)

# test_dataset = dataset.iloc[test_ids, :]
# test_dataset.to_csv('../Data/Processed/test_dataset.csv', index=False)

# train_ids = ~np.isin(np.where(all_ids)[0], test_ids)
# train_dataset = dataset.iloc[train_ids, :]
# test_dataset.to_csv('../Data/Processed/train_dataset.csv', index=False)

ids = np.arange(dataset.shape[0])
train_ids, test_ids = train_test_split(ids, test_size=0.1, random_state=50)

train_dataset = dataset.iloc[train_ids]
test_dataset = dataset.iloc[test_ids]

train_dataset.to_csv('../Data/Processed/train_dataset.csv')
test_dataset.to_csv('../Data/Processed/test_dataset.csv')

In [20]:
train_dataset

,Student_ID,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
532,STU0533,1,6.9,16.6,2.63,5.8,10.7,High,4.4,7.9,84.5,No,Average,No,Low
1648,STU1649,1,5.8,11.5,2.08,6.6,9.2,High,6.8,7.6,80.2,No,Poor,No,Medium
1574,STU1575,1,7.0,16.2,2.32,4.9,7.4,Moderate,9.4,4.7,80.0,No,Average,Yes,High
780,STU0781,2,8.2,24.0,2.42,5.0,8.9,Moderate,NaN,6.9,78.6,Yes,Poor,No,Medium
1207,STU1208,2,8.8,19.4,2.89,NaN,8.3,Moderate,8.5,4.2,95.7,No,Average,Yes,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313,STU1314,3,6.5,27.2,2.80,4.2,6.3,Moderate,5.7,7.2,83.9,No,Average,Yes,Medium
109,STU0110,3,NaN,21.3,2.00,6.7,10.1,Moderate,5.3,9.3,75.8,No,Average,No,Medium
1931,STU1932,1,7.8,12.7,2.91,8.6,8.7,Low,2.7,4.3,90.0,No,Poor,No,Medium
1504,STU1505,3,7.8,24.2,3.06,4.7,6.9,High,7.3,NaN,75.7,No,Average,No,Low


In [22]:
# def get_ages_studied_years_dist(dataset: pd.DataFrame, norm_age = 23):

#     min_age = dataset['Age'].min()
#     years = norm_age - min_age
#     fig, axes = plt.subplots(years, figsize=(10, 12))

#     for id, year in enumerate(range(min_age, norm_age)):

#         sns.barplot(data=dataset['Year_of_Study'][dataset['Age'] == year].value_counts(), ax=axes[id])
#         axes[id].set_title(f'{year} Studied Years Dist')
        
#     plt.tight_layout()

# get_ages_studied_years_dist(train_dataset)

In [ ]:
def process_non_binary_gender(dataset: pd.DataFrame, drop=True):
    if drop:
        indices = dataset[dataset['Gender'] == 'Non-binary'].index
        dataset = dataset.drop(indices)
    print(f'{(dataset['Gender'] == 'Non-binary').sum()} of non binary gender exists in data')

# process_non_binary_gender(train_dataset)

0 of non binary gender exists in data


In [ ]:
colum

Student_ID                        0
Year_of_Study                     0
Sleep_Hours                       0
Study_Hours_Per_Week              0
GPA                               0
Social_Media_Hours                0
Screen_Time_Hours                 0
Physical_Activity                 0
Financial_Stress                  0
Family_Support                    0
Attendance_Rate                   0
Part_Time_Job                     0
Diet_Quality                      0
Previous_Mental_Health_History    0
Mental_Health_Risk                0
dtype: int64

In [121]:
imputer = IterativeImputer()
after_imputation = imputer.fit_transform(train_dataset.select_dtypes(include='number'))

In [ ]:
train_dataset.loc[train_dataset['Social_Media_Hours'] ]

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64

In [61]:
def get_high_risk_val():
    new_dataset = dataset[dataset['Mental_Health_Risk'] == 'High']
    display(new_dataset.select_dtypes(include='number').corr())

get_high_risk_val()

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate
Age,1.000000,0.018435,0.048553,-0.007056,-0.015724,-0.062452,-0.039581,-0.020739,-0.003162,-0.030232
Year_of_Study,0.018435,1.000000,-0.050801,0.718318,0.321004,-0.329598,-0.217967,0.019647,-0.049699,0.019653
Sleep_Hours,0.048553,-0.050801,1.000000,-0.104782,0.112759,0.058802,0.125829,0.223069,-0.184800,0.020550
Study_Hours_Per_Week,-0.007056,0.718318,-0.104782,1.000000,0.417573,-0.474891,-0.317559,0.025467,0.004948,0.113851
GPA,-0.015724,0.321004,0.112759,0.417573,1.000000,-0.167128,-0.132051,0.128553,-0.104787,0.252767
Social_Media_Hours,-0.062452,-0.329598,0.058802,-0.474891,-0.167128,1.000000,0.684803,-0.103358,0.043627,-0.096963
Screen_Time_Hours,-0.039581,-0.217967,0.125829,-0.317559,-0.132051,0.684803,1.000000,-0.073658,0.023024,-0.059972
Financial_Stress,-0.020739,0.019647,0.223069,0.025467,0.128553,-0.103358,-0.073658,1.000000,0.156577,0.054031
Family_Support,-0.003162,-0.049699,-0.184800,0.004948,-0.104787,0.043627,0.023024,0.156577,1.000000,-0.103134
Attendance_Rate,-0.030232,0.019653,0.020550,0.113851,0.252767,-0.096963,-0.059972,0.054031,-0.103134,1.000000


In [47]:
for column in dataset.columns:
    if dataset[column].dtype == 'float64' or dataset[column].dtype == 'int64':
        new_val = dataset[column].mean()
    else:  new_val = dataset[column].mode().iloc[0]
    dataset[column] = dataset[column].fillna(new_val)

In [48]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             2000 non-null   int64  
 1   Gender                          2000 non-null   str    
 2   Year_of_Study                   2000 non-null   int64  
 3   Sleep_Hours                     2000 non-null   float64
 4   Study_Hours_Per_Week            2000 non-null   float64
 5   GPA                             2000 non-null   float64
 6   Social_Media_Hours              2000 non-null   float64
 7   Screen_Time_Hours               2000 non-null   float64
 8   Physical_Activity               2000 non-null   str    
 9   Financial_Stress                2000 non-null   float64
 10  Family_Support                  2000 non-null   float64
 11  Attendance_Rate                 2000 non-null   float64
 12  Part_Time_Job                   2000 non-null